In [2]:
import os
import sys
import json
import pickle
import warnings
import numpy as np
import sympy as sp
import pandas as pd
import proplot as pplt
from IPython.display import display, HTML
sys.path.insert(0,'..')
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [3]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
MODELSDIR = CONFIGS['filepaths']['models']
SRCONFIG  = CONFIGS['experiments']['sr']
SEEDS     = SRCONFIG['seeds']

In [4]:
VARDICT = {
    'rh':       r'\widehat{\mathrm{RH}}',
    'thetae':   r'\widehat{\theta}_{e}',
    'thetaestar':r'{\widehat{\theta}_{e}^{*}}',
    'bl':       r'\mathrm{B_L}',
    'lf':       r'\mathrm{LF}',
    'shf':      r'\mathrm{SHF}',
    'lhf':      r'\mathrm{LHF}'}

SYMBOLS  = {k: sp.Symbol(k) for k in VARDICT}

FUNCDICT = {
    'cube':   lambda x: x**3,
    'square': lambda x: x**2,
    'sqrt':   sp.sqrt,
    'abs':    sp.Abs,
    'neg':    lambda x: -x,
    'exp':    sp.exp,
    'log':    sp.log,
    'sin':    sp.sin,
    'cos':    sp.cos,
    'max':    sp.Max,
    'min':    sp.Min}

TERMORDER = {'bl':0,'rh':1,'thetae':2,'thetaestar':3,'lf':4,'shf':5,'lhf':6}

def _to_sympy_expr(eq):
    return sp.sympify(eq, locals={**SYMBOLS, **FUNCDICT})

def _round_numbers(expr, ndigits=4):
    return expr.xreplace({n: sp.Float(round(float(n), ndigits), ndigits) for n in expr.atoms(sp.Float)})

def _term_key(term):
    symbols = term.free_symbols
    if not symbols:
        return (99, str(term))
    names = sorted(s.name for s in symbols)
    return (min(TERMORDER.get(n, 50) for n in names), str(term))

def _ordered_add_terms(expr):
    if isinstance(expr, sp.Add):
        terms = sp.Add.make_args(expr)
        return sp.Add(*sorted(terms, key=_term_key), evaluate=False)
    return expr

def _order_expr(expr):
    if expr.args:
        expr = expr.func(*[_order_expr(arg) for arg in expr.args], evaluate=False)
    if isinstance(expr, sp.Add):
        expr = _ordered_add_terms(expr)
    return expr

def _latex_expr(expr):
    symbolnames = {SYMBOLS[k]: v for k, v in VARDICT.items()}
    latex = sp.latex(expr, symbol_names=symbolnames, mul_symbol='dot')
    latex = latex.replace(r'\left', '').replace(r'\right', '')
    latex = ' '.join(latex.split())
    return latex

def prettify(eq):
    try:
        expr = _to_sympy_expr(str(eq).strip())
        expr = sp.expand(expr)
        expr = _round_numbers(expr, ndigits=4)
        expr = _order_expr(expr)
        return '$' + _latex_expr(expr) + '$'
    except Exception:
        return str(eq).strip()

def load_equations(runname):
    seedframes = {}
    for seed in SEEDS:
        filepath = os.path.join(MODELSDIR, 'sr', f'{runname}_{seed}_equations.csv')
        if not os.path.exists(filepath):
            continue
        df = pd.read_csv(filepath)
        df['seed'] = seed
        seedframes[seed] = df
    return seedframes

In [5]:
# COLORS = ['#5BA7DA', '#F2C85E', '#D42028']

# for runname in SRCONFIG['runs']:
#     seedframes = load_equations(runname)
#     if not seedframes:
#         continue
#     selcomplexity = [spec['refcomplexity'] for spec in SRCONFIG['optimizedeqs'].values() if spec['runfrom']==runname]
#     allcomplexity = pd.concat(seedframes.values())['complexity']
#     fig,ax = pplt.subplots(refwidth=5,refheight=2)
#     ax.format(title=runname,xlabel='Complexity',xlim=(0,allcomplexity.max()+1),ylabel='Loss')
#     for i,(seed,df) in enumerate(sorted(seedframes.items())):
#         df    = df.sort_values('complexity')
#         color = COLORS[i%len(COLORS)]
#         ax.plot(df['complexity'],df['loss'],color=color,alpha=0.5,linewidth=1,linestyle='--',zorder=1,label='')
#         ax.scatter(df['complexity'],df['loss'],color=color,marker='.',zorder=3,label=f'Seed {seed}')
#         for complexity in selcomplexity:
#             row = df[df['complexity']==complexity]
#             if not row.empty:
#                 ax.scatter([row['complexity'].values[0]],[row['loss'].values[0]],
#                            color=color,edgecolors='k',marker='*',markersize=100,linewidths=0.5,zorder=5)
#     ax.scatter([],[],color='k',marker='*',markersize=100,label='Selected')
#     ax.legend(loc='ur',ncols=1)
#     pplt.show()

In [6]:
def load_registry():
    pklpath = os.path.join(MODELSDIR, 'sr', 'optimized_equations.pkl')
    if not os.path.exists(pklpath):
        return {}
    with open(pklpath, 'rb') as f:
        return pickle.load(f)

def prettify_optimized(opt):
    ns = {**SYMBOLS, **FUNCDICT}
    ns.update({k: sp.Float(v) for k, v in opt['constants'].items()})
    try:
        expr = eval(opt['form'], {'__builtins__': {}}, ns)
        expr = sp.expand(expr)
        expr = _round_numbers(expr, ndigits=4)
        expr = _order_expr(expr)
        return r'$\hat{P} = ' + _latex_expr(expr) + '$'
    except Exception as e:
        return f"{opt['form']}  [render error: {e}]"

registry = load_registry()
rows = []
for name, eqspec in SRCONFIG['optimizedeqs'].items():
    opt = registry.get(name)
    if opt is None:
        rows.append({'Model': eqspec['description'], 
                     #'Form': eqspec['form'],
                     'Train MSE': '—', 'Valid MSE': '—'})
    else:
        rows.append({'Model':     eqspec['description'],
                     'Equation':  prettify_optimized(opt),
                     'Train MSE': f"{opt['train_loss']:.4f}",
                     'Valid MSE': f"{opt['valid_loss']:.4f}"})
display(HTML(pd.DataFrame(rows).to_html(escape=False,index=False)))

Model,Train MSE,Valid MSE
SR-BL,—,—
SR-LO,—,—
SR-MED,—,—
SR-HI,—,—
SR-FULL,—,—


## Physical-space constants

Analytically expand the optimized standardized constants into named physical-space constants. Each equation is written as:

$$P\ [\mathrm{mm}] = \max\!\left(\mathrm{expm1}\!\left(\max(0,\ f(\mathbf{X}))\right),\ 0\right)$$

where $f$ is dimensionless (it equals $\log(1 + P_{\mathrm{mm}})$ at the optimum), so scale coefficients have units of $(\mathrm{input\ unit})^{-\mathrm{power}}$ and intercepts are dimensionless.

In [36]:
statsfile = os.path.join('..', 'data', 'splits', 'stats.json')
if os.path.exists(statsfile):
    with open(statsfile, 'r', encoding='utf-8') as f:
        STATS = json.load(f)
    print('Loaded stats.json')
else:
    STATS = None
    print('[warn] stats.json not found — run on the cluster to populate this table')

Loaded stats.json


In [37]:
def _phys_sr_bl(constants, stats, targetvar='tp'):
    """a * cube(bl + b) + c  →  α_BL·(BL − BL_crit)³ + β_BL"""
    a, b, c  = constants['a'], constants['b'], constants['c']
    mu_bl    = stats['bl_mean'];    sig_bl = stats['bl_std']
    sig_tp   = stats[f'{targetvar}_std']; mu_tp = stats[f'{targetvar}_mean']
    return [
        ('BL_crit', mu_bl - b*sig_bl,           'm s⁻²',       'Critical BL for onset'),
        ('α_BL',    sig_tp*a / sig_bl**3,        '(m s⁻²)⁻³',   'Cubic sensitivity'),
        ('β_BL',    sig_tp*c + mu_tp,            '—',            'Intercept'),
    ]

def _phys_sr_lo(constants, stats, targetvar='tp'):
    """a * exp(b * rh) + c  →  α_LO·exp(B_LO·RH) + β_LO"""
    a, b, c  = constants['a'], constants['b'], constants['c']
    mu_rh    = stats['rh_mean'];    sig_rh = stats['rh_std']
    sig_tp   = stats[f'{targetvar}_std']; mu_tp = stats[f'{targetvar}_mean']
    return [
        ('α_LO', sig_tp*a*np.exp(-b*mu_rh/sig_rh), '—',     'Exponential prefactor'),
        ('B_LO', b/sig_rh,                          '%⁻¹',   'Exponential RH sensitivity'),
        ('β_LO', sig_tp*c + mu_tp,                  '—',     'Intercept'),
    ]

def _phys_sr_med(constants, stats, targetvar='tp'):
    """a * cube(max(rh, thetae + b*thetaestar + c))"""
    a, b, c  = constants['a'], constants['b'], constants['c']
    mu_te    = stats['thetae_mean'];     sig_te  = stats['thetae_std']
    mu_tes   = stats['thetaestar_mean']; sig_tes = stats['thetaestar_std']
    sig_rh   = stats['rh_std']
    sig_tp   = stats[f'{targetvar}_std']
    kappa      = -b * sig_te / sig_tes
    theta_crit = mu_te + b*(sig_te/sig_tes)*mu_tes - c*sig_te
    return [
        ('α_MED',     sig_tp*a / sig_te**3,  'K⁻³',  'Cubic sensitivity'),
        ('κ',         kappa,                  '—',     'θe* weighting (≈ 1 → CAPE proxy)'),
        ('Θe_crit',   theta_crit,             'K',     'Critical buoyancy deficit for onset'),
        ('σΘe/σRH',   sig_te/sig_rh,          'K %⁻¹', 'RH → RH_eff scale factor'),
    ]

def _phys_sr_hi(constants, stats, targetvar='tp'):
    """(a - b*lhf) * cube(max(rh, thetae + c*thetaestar + d))"""
    a, b, c, d = constants['a'], constants['b'], constants['c'], constants['d']
    mu_te    = stats['thetae_mean'];     sig_te  = stats['thetae_std']
    mu_tes   = stats['thetaestar_mean']; sig_tes = stats['thetaestar_std']
    mu_lhf   = stats['lhf_mean'];        sig_lhf = stats['lhf_std']
    sig_rh   = stats['rh_std']
    sig_tp   = stats[f'{targetvar}_std']
    kappa      = -c * sig_te / sig_tes
    theta_crit = mu_te + c*(sig_te/sig_tes)*mu_tes - d*sig_te
    return [
        ('α_HI',    sig_tp / sig_te**3,        'K⁻³',   'Cubic sensitivity'),
        ('A_LHF',   a + b*mu_lhf/sig_lhf,      '—',     'LHF flux offset'),
        ('γ_LHF',   b/sig_lhf,                  'W⁻¹ m²','LHF sensitivity'),
        ('κ',       kappa,                       '—',     'θe* weighting'),
        ('Θe_crit', theta_crit,                  'K',     'Critical buoyancy deficit for onset'),
        ('σΘe/σRH', sig_te/sig_rh,               'K %⁻¹', 'RH → RH_eff scale factor'),
    ]

PHYS_FNS = {'sr_bl': _phys_sr_bl, 'sr_lo': _phys_sr_lo,
            'sr_med': _phys_sr_med, 'sr_hi': _phys_sr_hi}

In [38]:
if STATS is None or not registry:
    print('[warn] Need stats.json and optimized_equations.pkl — run on the cluster')
else:
    targetvar = CONFIGS['variables']['target']
    all_rows  = []
    for name, eqspec in SRCONFIG['optimizedeqs'].items():
        opt = registry.get(name)
        fn  = PHYS_FNS.get(name)
        if opt is None or fn is None:
            continue
        for sym, val, units, desc in fn(opt['constants'], STATS, targetvar=targetvar):
            all_rows.append({
                'Model':   eqspec['description'],
                'Symbol':  sym,
                'Value':   f'{val:.4g}',
                'Units':   units,
                'Description': desc,
            })
    display(HTML(pd.DataFrame(all_rows).to_html(escape=False, index=False)))

Model,Symbol,Value,Units,Description
SR-BL,BL_crit,0.01957,m s⁻²,Critical BL for onset
SR-BL,α_BL,7452,(m s⁻²)⁻³,Cubic sensitivity
SR-BL,β_BL,1.699,—,Intercept
SR-LO,α_LO,0.1829,—,Exponential prefactor
SR-LO,B_LO,0.02754,%⁻¹,Exponential RH sensitivity
SR-LO,β_LO,-0.7804,—,Intercept
SR-MED,α_MED,0.0003087,K⁻³,Cubic sensitivity
SR-MED,κ,1.14,—,θe* weighting (≈ 1 → CAPE proxy)
SR-MED,Θe_crit,-63.45,K,Critical buoyancy deficit for onset
SR-MED,σΘe/σRH,0.4137,K %⁻¹,RH → RH_eff scale factor


### Gaussian kernel parameters (nn_gauss)

Kernel center and width in sigma-level coordinates (σ ∈ [0.50, 1.00], surface = 1.00).
Conversion from learned parameters: σ_center = 0.75 + μ × 0.25, Δσ = exp(logstd) × 0.25.

In [39]:
import torch

nn_cfg    = CONFIGS['experiments']['nn']
fieldvars = nn_cfg['runs']['nn_gauss']['fieldvars']
modeldir  = os.path.join('..', CONFIGS['filepaths']['models'].split('monsoon-discovery/')[-1], 'nn')
seeds     = nn_cfg['seeds']

kernel_rows = []
for seed in seeds:
    ckpt_path = os.path.join(modeldir, f'nn_gauss_{seed}.pth')
    if not os.path.exists(ckpt_path):
        continue
    state = torch.load(ckpt_path, map_location='cpu')
    if 'state_dict' in state:
        state = state['state_dict']
    mu     = state['kernel.function.mu'].numpy()
    logstd = state['kernel.function.logstd'].numpy()
    for i, fv in enumerate(fieldvars):
        kernel_rows.append({
            'Seed':        seed,
            'Variable':    fv,
            'σ_center':    round(float(0.75 + mu[i]*0.25), 4),
            'Δσ (1σ width)': round(float(np.exp(logstd[i])*0.25), 4),
        })

if not kernel_rows:
    print('[warn] No nn_gauss checkpoints found — run on the cluster')
else:
    kdf = pd.DataFrame(kernel_rows)
    summary = kdf.groupby('Variable')[['σ_center','Δσ (1σ width)']].agg(['mean','std']).round(4)
    summary.columns = ['σ_center (mean)', 'σ_center (std)', 'Δσ mean', 'Δσ std']
    display(HTML(summary.to_html()))

,σ_center (mean),σ_center (std),Δσ mean,Δσ std
Variable,,,,
rh,-0.0410,0.1772,0.5374,0.0445
thetae,1.0198,0.0075,0.1459,0.0054
thetaestar,0.7707,0.0016,0.1279,0.0007
